# Tail Risk y Valor en Riesgo (VaR) con Machine Learning · SPY 2005–2025

<u>Estudios</u>: **Máster Universitario en Inteligencia Artificial** · UDIT
<u>Asignatura</u>: **Lenguajes de Programación para IA**
<u>Alumno</u>: **Álvaro Manuel Balegas López**

---

### Objetivo

Los modelos clásicos de riesgo suponen que los retornos son gaussianos y estacionarios. En la realidad, los mercados tienen **colas gruesas** y **la volatilidad se agrupa en el tiempo**: justo en las crisis, cuando más importa, el VaR clásico **infravalora** el riesgo.

En este notebook:

1. **EDA**: demostramos con datos del S&P 500 (ETF SPY) que los retornos no son normales y que la volatilidad tiene memoria.
2. **Simulación de crisis (Monte Carlo)**: comparamos la probabilidad de un *crash* mensual con distintos generadores de escenarios y hacemos un *stress test* remuestreando los días del COVID-19.
3. **Modelo de ML**: entrenamos un **Gradient Boosting con pérdida cuantílica** que estima directamente el cuantil condicional del 5 %, es decir, el VaR al 95 %.
4. **Backtesting**: lo comparamos con los métodos de la industria (histórico, normal y EWMA RiskMetrics) mediante los tests de **Kupiec** y **Christoffersen**.
5. **Exportamos el modelo** para usarlo desde `var_predict.py` por línea de comandos.

## 0. Fundamentos matemáticos

### Valor en Riesgo

Sea $r_{t+1} = \ln(P_{t+1}/P_t)$ el retorno logarítmico de mañana y $\mathcal{F}_t$ la información disponible hoy. El **VaR al nivel de confianza $1-\alpha$** es la pérdida que solo se supera con probabilidad $\alpha$:

$$
P\left(r_{t+1} < -\text{VaR}_{t+1} \mid \mathcal{F}_t\right) = \alpha
\quad\Longleftrightarrow\quad
\text{VaR}_{t+1} = -\,q_\alpha\!\left(r_{t+1}\mid\mathcal{F}_t\right)
$$

Es decir, **el VaR es un cuantil condicional**. Usamos $\alpha = 0.05$ (VaR 95 %).

### Por qué fallan los métodos clásicos

| Método | Fórmula | Problema |
|---|---|---|
| Normal | $\hat q_\alpha = \hat\mu + z_\alpha\hat\sigma$ | Supone colas gaussianas → infravalora eventos extremos |
| Histórico | cuantil empírico de los últimos 250 días | Reacciona tarde: la ventana "recuerda" la calma previa |
| EWMA (RiskMetrics) | $\sigma^2_t = \lambda\sigma^2_{t-1} + (1-\lambda) r_t^2$ | Se adapta, pero mantiene la hipótesis normal |

### Regresión cuantílica: aprender el cuantil directamente

En lugar de suponer una distribución, aprendemos la función $\hat q_\alpha(\mathbf{x}_t)$ minimizando la **pinball loss**:

$$
L_\alpha(y, \hat q) =
\begin{cases}
\alpha\,(y-\hat q) & \text{si } y \ge \hat q \\
(1-\alpha)\,(\hat q - y) & \text{si } y < \hat q
\end{cases}
$$

El minimizador de $\mathbb{E}[L_\alpha(Y, q)]$ es exactamente el cuantil $\alpha$ de $Y$. Con $\alpha=0.05$, pasarse por debajo (una violación) penaliza 19 veces más que quedarse por encima. Con un modelo de árboles (*Gradient Boosting*), el cuantil puede depender de forma no lineal de la volatilidad reciente, el *drawdown*, etc.

## 1. Preparación de entorno

In [14]:
import sys, platform
print("Python:", sys.version)
print("Executable:", sys.executable)

Python: 3.12.7 (v3.12.7:0b05ead877f, Sep 30 2024, 23:18:00) [Clang 13.0.0 (clang-1300.0.29.30)]
Executable: /usr/local/bin/python3


In [15]:
import os
import warnings

import numpy as np
import pandas as pd
from scipy import stats

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

#### PARA EL RENDER DE LOS GRAFICOS ####
pio.renderers.default = "notebook_connected"
pio.templates.default = "plotly_white"

warnings.filterwarnings("ignore")
SEED = 42
rng = np.random.default_rng(SEED)

ALPHA = 0.05   # VaR al 95 %: cuantil del 5 % de la distribución de retornos

## 2. Carga de datos

Descargamos precios diarios del **SPY** (ETF que replica el S&P 500) entre 2005 y 2025 con `yfinance`. El periodo incluye la crisis financiera de 2008, la crisis de deuda europea de 2011, el crash del COVID-19 en 2020 y el *bear market* de 2022.

Usamos precios **ajustados** (`auto_adjust=True`) para que dividendos y *splits* no generen retornos falsos. Al ejecutarse por primera vez, la celda guarda el CSV en `data/spy.csv` (~200 KB) y, a partir de ahí, el notebook funciona sin conexión.

In [16]:
TICKER = "SPY"
START, END = "2005-01-01", "2025-12-31"
CSV_PATH = os.path.join("data", "spy.csv")

if not os.path.exists(CSV_PATH):
    import yfinance as yf  # pip install yfinance
    os.makedirs("data", exist_ok=True)
    raw = yf.download(TICKER, start=START, end=END, auto_adjust=True, progress=False)
    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = raw.columns.get_level_values(0)
    raw[["Close", "Volume"]].rename_axis("Date").to_csv(CSV_PATH)

prices = pd.read_csv(CSV_PATH, parse_dates=["Date"], index_col="Date")
prices = prices[["Close", "Volume"]].astype(float).sort_index()
print(f"{len(prices):,} sesiones · {prices.index.min().date()} → {prices.index.max().date()}")
prices.head()

5,282 sesiones · 2005-01-03 → 2025-12-30


,Close,Volume
Date,,
2005-01-03,80.973534,55748000.0
2005-01-04,79.984100,69167600.0
2005-01-05,79.432137,65667300.0
2005-01-06,79.835991,47814700.0
2005-01-07,79.721573,55847700.0


In [17]:
# Calidad del dato
print("Valores nulos por columna:\n", prices.isna().sum(), sep="")
print("\nFechas duplicadas:", prices.index.duplicated().sum())
gaps = prices.index.to_series().diff().dt.days
print("Mayor hueco entre sesiones (días naturales):", int(gaps.max()))
prices.describe().T

Valores nulos por columna:
Close     0
Volume    0
dtype: int64

Fechas duplicadas: 0
Mayor hueco entre sesiones (días naturales): 5


,count,mean,std,min,25%,50%,75%,max
Close,5282.0,2.237295e+02,1.553942e+02,4.955751e+01,9.772138e+01,1.702963e+02,3.120926e+02,6.850295e+02
Volume,5282.0,1.231195e+08,9.010862e+07,1.527000e+07,6.520338e+07,9.226850e+07,1.510283e+08,8.710263e+08


In [18]:
# Retornos logarítmicos diarios y periodos de crisis para anotar los gráficos
r = np.log(prices["Close"]).diff().dropna()
r.name = "ret"

CRISES = {
    "Crisis financiera 2008": ("2007-10-09", "2009-03-09"),
    "Deuda europea 2011":     ("2011-07-22", "2011-10-03"),
    "COVID-19":               ("2020-02-19", "2020-03-23"),
    "Bear market 2022":       ("2022-01-03", "2022-10-12"),
}

def shade_crises(fig, row=None, col=None, label=True):
    # Sombrea los periodos de crisis en una figura Plotly
    for name, (x0, x1) in CRISES.items():
        fig.add_vrect(
            x0=x0, x1=x1, fillcolor="crimson", opacity=0.12, line_width=0,
            annotation_text=name if label else None,
            annotation_position="top left", annotation_font_size=10,
            row=row, col=col,
        )
    return fig

## 3. Análisis exploratorio (EDA)

### 3.1 Evolución del precio y crisis

In [19]:
fig = px.line(prices, y="Close", log_y=True,
              title="SPY · precio ajustado (escala logarítmica) y periodos de crisis")
shade_crises(fig)
fig.update_layout(xaxis=dict(rangeslider=dict(visible=True)),
                  yaxis_title="Precio (USD, log)", xaxis_title=None, showlegend=False)
fig.show()

La escala logarítmica es la adecuada aquí: en ella, una misma distancia vertical equivale al mismo **porcentaje** de variación, así que la caída de 2008 (≈ −55 %) se puede comparar visualmente con la de 2020 (≈ −34 %).

### 3.2 Retornos diarios: ¿son normales?

In [20]:
fig = px.line(r, title="Retornos logarítmicos diarios")
shade_crises(fig, label=False)
fig.update_layout(showlegend=False, yaxis_tickformat=".0%", xaxis_title=None, yaxis_title="r_t")
fig.show()

In [21]:
n = len(r)
sigma = r.std()
k_sigma = 4
obs_extremos = int((np.abs(r - r.mean()) > k_sigma * sigma).sum())
esp_extremos = n * 2 * stats.norm.sf(k_sigma)
jb_stat, jb_p = stats.jarque_bera(r)

summary = pd.Series({
    "Observaciones": n,
    "Media diaria (%)": 100 * r.mean(),
    "Volatilidad diaria (%)": 100 * sigma,
    "Volatilidad anualizada (%)": 100 * sigma * np.sqrt(252),
    "Asimetría (skew)": stats.skew(r),
    "Curtosis en exceso": stats.kurtosis(r),          # normal = 0
    "Peor día (%)": 100 * r.min(),
    "Mejor día (%)": 100 * r.max(),
    f"Días con |r| > {k_sigma}σ observados": obs_extremos,
    f"Días con |r| > {k_sigma}σ esperados si fuera normal": esp_extremos,
    "Jarque-Bera p-valor": jb_p,
}).round(4)
summary.to_frame("SPY")

,SPY
Observaciones,5281.0000
Media diaria (%),0.0403
Volatilidad diaria (%),1.2012
Volatilidad anualizada (%),19.0691
Asimetría (skew),-0.3051
Curtosis en exceso,14.6941
Peor día (%),-11.5887
Mejor día (%),13.5578
Días con |r| > 4σ observados,40.0000
Días con |r| > 4σ esperados si fuera normal,0.3345


**Lectura:**

- La **curtosis en exceso** sale muy por encima de 0, que es el valor de la normal: las colas son gruesas.
- La **asimetría es negativa**: las caídas fuertes son más frecuentes que las subidas fuertes.
- El número de días con movimientos de más de 4σ es **decenas de veces mayor** de lo que predice una normal.
- El test de **Jarque-Bera** rechaza la normalidad con un p-valor prácticamente nulo.

Lo vemos gráficamente: el histograma en escala logarítmica muestra las colas que la normal no capta.

In [22]:
grid = np.linspace(r.min(), r.max(), 500)
nu_t, loc_t, scale_t = stats.t.fit(r)
pdf_norm = stats.norm.pdf(grid, r.mean(), sigma)
pdf_t = stats.t.pdf(grid, nu_t, loc_t, scale_t)

fig = make_subplots(rows=1, cols=2, subplot_titles=("Densidad (escala lineal)", "Densidad (escala log): las colas"))
for col in (1, 2):
    fig.add_trace(go.Histogram(x=r, nbinsx=200, histnorm="probability density", name="Empírico",
                               marker_color="lightslategray", showlegend=(col == 1)), row=1, col=col)
    fig.add_trace(go.Scatter(x=grid, y=pdf_norm, name="Normal", line=dict(color="crimson"),
                             showlegend=(col == 1)), row=1, col=col)
    fig.add_trace(go.Scatter(x=grid, y=pdf_t, name=f"Student-t (ν={nu_t:.1f})",
                             line=dict(color="royalblue", dash="dash"), showlegend=(col == 1)), row=1, col=col)
fig.update_yaxes(type="log", range=[-3, 2], row=1, col=2)
fig.update_xaxes(tickformat=".0%")
fig.update_layout(title="Distribución de retornos: Normal vs Student-t", bargap=0)
fig.show()

In [23]:
# QQ-plot frente a la normal: si los datos fueran normales, los puntos caerían sobre la recta
(osm, osr), (slope, intercept, _) = stats.probplot(r, dist="norm")
fig = go.Figure()
fig.add_trace(go.Scatter(x=osm, y=osr, mode="markers", name="Cuantiles empíricos",
                         marker=dict(size=4, color="lightslategray")))
fig.add_trace(go.Scatter(x=osm, y=slope * osm + intercept, mode="lines", name="Normal teórica",
                         line=dict(color="crimson")))
fig.update_layout(title="QQ-plot de retornos vs Normal", xaxis_title="Cuantiles teóricos N(0,1)",
                  yaxis_title="Cuantiles empíricos", yaxis_tickformat=".0%")
fig.show()

La forma de **"S" invertida** del QQ-plot es la firma de las colas gruesas: los cuantiles empíricos extremos están mucho más lejos que los teóricos, sobre todo en la cola izquierda (pérdidas).

### 3.3 Volatility clustering: la volatilidad tiene memoria

Si los retornos fueran i.i.d., ni $r_t$ ni $|r_t|$ tendrían autocorrelación. Lo comprobamos con la función de autocorrelación (ACF):

$$\rho(k) = \frac{\sum_t (x_t-\bar x)(x_{t+k}-\bar x)}{\sum_t (x_t-\bar x)^2}$$

In [24]:
def acf(x, nlags=60):
    x = np.asarray(x) - np.mean(x)
    denom = np.sum(x**2)
    return np.array([np.sum(x[:-k] * x[k:]) / denom for k in range(1, nlags + 1)])

lags = np.arange(1, 61)
band = 1.96 / np.sqrt(len(r))

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=("ACF de r_t  (dirección: casi impredecible)",
                                    "ACF de |r_t|  (magnitud: memoria larga)"))
fig.add_trace(go.Bar(x=lags, y=acf(r), marker_color="lightslategray", name="r_t"), row=1, col=1)
fig.add_trace(go.Bar(x=lags, y=acf(np.abs(r)), marker_color="royalblue", name="|r_t|"), row=2, col=1)
for row in (1, 2):
    fig.add_hline(y=band, line_dash="dot", line_color="crimson", row=row, col=1)
    fig.add_hline(y=-band, line_dash="dot", line_color="crimson", row=row, col=1)
fig.update_layout(title="Autocorrelación: retornos vs retornos absolutos", showlegend=False, height=550)
fig.update_xaxes(title_text="Lag (días)", row=2, col=1)
fig.show()

**Esta es la observación clave del notebook.** La ACF de $r_t$ está casi dentro de la banda de confianza, así que **no se puede predecir la dirección**. En cambio, la ACF de $|r_t|$ es positiva y significativa durante más de 60 días: **los días turbulentos vienen seguidos de días turbulentos**. Por eso un VaR que use la volatilidad reciente debería superar a uno estático.

In [25]:
vol_20 = r.rolling(20).std() * np.sqrt(252)
drawdown = prices["Close"] / prices["Close"].cummax() - 1

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.06,
                    subplot_titles=("Volatilidad realizada 20d (anualizada)", "Drawdown desde máximos"))
fig.add_trace(go.Scatter(x=vol_20.index, y=vol_20, line=dict(color="royalblue"), name="Vol 20d"), row=1, col=1)
fig.add_trace(go.Scatter(x=drawdown.index, y=drawdown, fill="tozeroy", line=dict(color="crimson"),
                         name="Drawdown"), row=2, col=1)
shade_crises(fig, row="all", col=1, label=False)
fig.update_yaxes(tickformat=".0%")
fig.update_layout(title="Regímenes de volatilidad y caídas", height=600, showlegend=False)
fig.show()

In [26]:
df_year = r.to_frame()
df_year["Año"] = df_year.index.year.astype(str)
fig = px.box(df_year, x="Año", y="ret", points="outliers",
             title="Distribución de retornos diarios por año")
fig.update_layout(yaxis_tickformat=".0%", yaxis_title="r_t")
fig.show()

La dispersión cambia mucho entre años: 2008 y 2020 tienen cajas y bigotes mucho más anchos que 2017. **La distribución no es estacionaria**, lo que es otro argumento contra un VaR con parámetros fijos.

### 3.4 Features para el modelo

Construimos las variables explicativas usando **solo información disponible al cierre de $t$**. El target es $r_{t+1}$.

| Feature | Definición | Intuición |
|---|---|---|
| `ret_1d`, `ret_5d` | retorno de hoy y de la última semana | efecto apalancamiento: las caídas suben la volatilidad |
| `vol_5d`, `vol_20d`, `vol_60d` | desviación típica móvil | clustering a distintas escalas |
| `vol_ratio` | `vol_5d / vol_60d` | ¿se está calentando el régimen? |
| `drawdown` | caída desde el máximo de 252 días | estrés acumulado |
| `volume_z` | z-score del log-volumen (60 días) | pánico o capitulación |

> La misma función `build_features` está copiada en `var_predict.py`, para que el modelo reciba en inferencia exactamente las mismas variables con las que se entrenó.

In [27]:
FEATURES = ["ret_1d", "ret_5d", "vol_5d", "vol_20d", "vol_60d", "vol_ratio", "drawdown", "volume_z"]

def build_features(prices):
    # Features con información hasta el cierre de t; target = retorno del día siguiente
    df = pd.DataFrame(index=prices.index)
    close, volume = prices["Close"], prices["Volume"]
    r = np.log(close).diff()

    df["ret"] = r
    df["ret_1d"] = r
    df["ret_5d"] = r.rolling(5).sum()
    df["vol_5d"] = r.rolling(5).std()
    df["vol_20d"] = r.rolling(20).std()
    df["vol_60d"] = r.rolling(60).std()
    df["vol_ratio"] = df["vol_5d"] / df["vol_60d"]
    df["drawdown"] = close / close.rolling(252, min_periods=1).max() - 1
    logv = np.log(volume.replace(0, np.nan))
    df["volume_z"] = (logv - logv.rolling(60).mean()) / logv.rolling(60).std()

    df["target"] = r.shift(-1)
    return df

data = build_features(prices)
data[FEATURES + ["target"]].dropna().describe().T

,count,mean,std,min,25%,50%,75%,max
ret_1d,5221.0,0.000411,0.012061,-0.115887,-0.003943,0.000718,0.005801,0.135578
ret_5d,5221.0,0.002063,0.024393,-0.220564,-0.008026,0.004113,0.014768,0.177339
vol_5d,5221.0,0.009356,0.008036,0.000425,0.004666,0.007334,0.011418,0.092075
vol_20d,5221.0,0.009907,0.007026,0.002007,0.005814,0.008194,0.011627,0.059197
vol_60d,5221.0,0.010271,0.006383,0.003136,0.006785,0.008253,0.011766,0.047614
vol_ratio,5221.0,0.924343,0.465717,0.084040,0.583234,0.841011,1.177486,3.288198
drawdown,5221.0,-0.052669,0.083884,-0.514815,-0.064229,-0.018428,-0.003212,0.000000
volume_z,5221.0,-0.024057,1.091343,-4.450958,-0.789315,-0.096833,0.670585,4.524882
target,5221.0,0.000411,0.012061,-0.115887,-0.003943,0.000718,0.005801,0.135578


In [28]:
corr_df = data[FEATURES + ["target"]].dropna().copy()
corr_df["|target|"] = corr_df["target"].abs()
corr = corr_df.corr()

fig = px.imshow(corr, text_auto=".2f", color_continuous_scale="RdBu_r", zmin=-1, zmax=1,
                title="Matriz de correlación: features vs target (r_{t+1}) y |target|")
fig.update_layout(height=600)
fig.show()

Las features **apenas se correlacionan con `target`**, porque la dirección no es predecible, pero **sí con `|target|`**: las de volatilidad tienen correlaciones claramente positivas y el *drawdown*, negativa. El modelo no tiene que adivinar si mañana el mercado sube o baja. Tiene que estimar **cuánto puede llegar a caer**, que es justo lo que mide el VaR.

---

## 4. Simulación de crisis: Monte Carlo

Queremos responder a: **¿qué probabilidad hay de que el S&P 500 caiga más de un 20 % en un mes (21 sesiones)?** Generamos 10.000 escenarios con cuatro métodos:

1. **Normal i.i.d.**: la hipótesis de los libros de texto.
2. **Student-t i.i.d.**: colas gruesas ajustadas a los datos.
3. **Bootstrap histórico**: remuestreamos con reemplazo días reales de todo el periodo.
4. **Stress test · bootstrap COVID-19**: remuestreamos solo días de la crisis del COVID (19-feb a 30-abr de 2020). Responde a "¿y si el próximo mes se pareciera a marzo de 2020?".

Como usamos log-retornos, el retorno a 21 días de cada escenario es la suma de sus 21 retornos diarios: $R_{21} = \exp\left(\sum_{h=1}^{21} r_h\right) - 1$.

In [29]:
N_SIMS, H = 10_000, 21
STRESS_WINDOW = ("2020-02-19", "2020-04-30")
r_stress = r.loc[STRESS_WINDOW[0]:STRESS_WINDOW[1]].values

sims = {
    "Normal i.i.d.": rng.normal(r.mean(), sigma, (N_SIMS, H)),
    "Student-t i.i.d.": loc_t + scale_t * rng.standard_t(nu_t, (N_SIMS, H)),
    "Bootstrap histórico": rng.choice(r.values, (N_SIMS, H), replace=True),
    "Stress test (bootstrap COVID)": rng.choice(r_stress, (N_SIMS, H), replace=True),
}

# Frecuencia empírica histórica (ventanas solapadas de 21 sesiones)
emp_21 = np.exp(r.rolling(H).sum().dropna()) - 1

def tail_metrics(paths):
    r21 = np.exp(paths.sum(axis=1)) - 1
    r1 = np.exp(paths[:, 0]) - 1
    var99 = -np.quantile(r1, 0.01)
    es99 = -r1[r1 <= -var99].mean()
    return {"P(caída >20% en 21d) (%)": 100 * np.mean(r21 < -0.20),
            "P(caída >10% en 21d) (%)": 100 * np.mean(r21 < -0.10),
            "VaR 99% 1d (%)": 100 * var99,
            "Expected Shortfall 99% 1d (%)": 100 * es99}

mc = pd.DataFrame({k: tail_metrics(v) for k, v in sims.items()}).T
mc.loc["Histórico real (empírico)"] = pd.Series({
    "P(caída >20% en 21d) (%)": 100 * np.mean(emp_21 < -0.20),
    "P(caída >10% en 21d) (%)": 100 * np.mean(emp_21 < -0.10),
    "VaR 99% 1d (%)": -100 * np.quantile(np.exp(r) - 1, 0.01),
    "Expected Shortfall 99% 1d (%)": -100 * (np.exp(r[r <= r.quantile(0.01)]) - 1).mean(),
})
mc.round(3)

,P(caída >20% en 21d) (%),P(caída >10% en 21d) (%),VaR 99% 1d (%),Expected Shortfall 99% 1d (%)
Normal i.i.d.,0.000,1.850,2.793,3.188
Student-t i.i.d.,0.290,2.210,3.398,5.492
Bootstrap histórico,0.030,2.190,3.325,4.846
Stress test (bootstrap COVID),18.460,39.410,10.942,10.942
Histórico real (empírico),0.418,1.977,3.521,5.141


**Interpretación:**

- **Normal frente a Student-t:** la t ensancha la cola **a 1 día** (sube el VaR 99 % y, sobre todo, el *Expected Shortfall*). Pero a 21 días sus probabilidades de *crash* se parecen, porque al sumar 21 retornos independientes el Teorema Central del Límite devuelve la suma hacia una normal.
- **El bootstrap histórico** usa días reales, pero los baraja como si fueran independientes, y así pierde el volatility clustering. Por eso suele quedarse por debajo de la frecuencia real de caídas mensuales: en la realidad, los días malos llegan juntos.
- **El stress test** muestra que la probabilidad de *crash* depende del régimen en el que estemos. En un mes "tipo COVID" es órdenes de magnitud mayor que en un mes medio.

**Conclusión:** el riesgo de cola **no es una constante, depende del estado actual del mercado**. Esto motiva un VaR **condicional**, que es el modelo de la sección 5.

In [30]:
fig = go.Figure()
colors = {"Normal i.i.d.": "crimson", "Student-t i.i.d.": "orange",
          "Bootstrap histórico": "seagreen", "Stress test (bootstrap COVID)": "royalblue"}
for name, paths in sims.items():
    fig.add_trace(go.Histogram(x=np.exp(paths.sum(axis=1)) - 1, nbinsx=150, name=name,
                               histnorm="probability density", opacity=0.5,
                               marker_color=colors[name]))
fig.add_vline(x=-0.20, line_dash="dash", line_color="black", annotation_text="−20 %")
fig.update_layout(barmode="overlay", title="Distribución simulada del retorno a 21 sesiones",
                  xaxis_tickformat=".0%", yaxis_type="log", xaxis_title="Retorno acumulado 21d")
fig.show()

In [31]:
# Fan chart: abanico de trayectorias de precio (base 100), escenario normal vs stress test
fig = make_subplots(rows=1, cols=2, shared_yaxes=True,
                    subplot_titles=("Bootstrap histórico", "Stress test (bootstrap COVID)"))
days = np.arange(H + 1)
for col, key in enumerate(["Bootstrap histórico", "Stress test (bootstrap COVID)"], start=1):
    price_paths = 100 * np.exp(np.cumsum(np.hstack([np.zeros((N_SIMS, 1)), sims[key]]), axis=1))
    for i in range(60):
        fig.add_trace(go.Scatter(x=days, y=price_paths[i], mode="lines", showlegend=False,
                                 line=dict(width=0.6, color="lightgray")), row=1, col=col)
    for qq, dash in [(0.01, "dot"), (0.05, "dash"), (0.5, "solid"), (0.95, "dash")]:
        fig.add_trace(go.Scatter(x=days, y=np.quantile(price_paths, qq, axis=0), mode="lines",
                                 name=f"Percentil {int(qq * 100)}", showlegend=(col == 1),
                                 line=dict(color=colors[key], dash=dash, width=2)), row=1, col=col)
fig.update_layout(title="Escenarios simulados de precio a 1 mes (base 100)", height=450)
fig.update_xaxes(title_text="Sesiones")
fig.show()

---

## 5. Modelado: VaR condicional con regresión cuantílica

### 5.1 Benchmarks clásicos de VaR

Los tres métodos estándar de la industria, calculados solo con información disponible hasta $t$. Todos devuelven el **cuantil** $q_\alpha$, que es negativo; el VaR como pérdida positiva es $-q_\alpha$.

In [32]:
def var_historical(ret, window=250, alpha=ALPHA):
    # Simulación histórica: cuantil empírico de la ventana móvil
    return ret.rolling(window).quantile(alpha)

def var_normal(ret, window=250, alpha=ALPHA):
    # Paramétrico gaussiano: mu + z_alpha * sigma con ventana móvil
    return ret.rolling(window).mean() + stats.norm.ppf(alpha) * ret.rolling(window).std()

def var_ewma(ret, lam=0.94, alpha=ALPHA):
    # RiskMetrics (J.P. Morgan, 1996): sigma²_t = lam·sigma²_{t-1} + (1-lam)·r²_t
    sigma2 = (ret**2).ewm(alpha=1 - lam, adjust=False).mean()
    return stats.norm.ppf(alpha) * np.sqrt(sigma2)

data["q_hist"] = var_historical(data["ret"])
data["q_normal"] = var_normal(data["ret"])
data["q_ewma"] = var_ewma(data["ret"])

### 5.2 Train/test cronológico

Con series temporales **no se puede usar `train_test_split` aleatorio**: el modelo vería datos del futuro (*look-ahead bias*). Partimos por fecha:

- **Train:** 2005–2017, que incluye la crisis de 2008 y la de 2011.
- **Test:** 2018–2025, completamente fuera de muestra, con el COVID y 2022 incluidos.

Quitamos la última fila de train, porque su target $r_{t+1}$ ya cae en 2018.

In [33]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import QuantileRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV, cross_val_score
from sklearn.metrics import mean_pinball_loss, make_scorer
import joblib

model_df = data.dropna(subset=FEATURES + ["target", "q_hist", "q_normal", "q_ewma"])

TRAIN_END = "2017-12-31"
train = model_df.loc[:TRAIN_END].iloc[:-1].copy()
test = model_df.loc["2018-01-01":].copy()

X_train, y_train = train[FEATURES], train["target"]
X_test, y_test = test[FEATURES], test["target"]
print(f"Train: {len(train):,} filas ({train.index.min().date()} → {train.index.max().date()})")
print(f"Test : {len(test):,} filas ({test.index.min().date()} → {test.index.max().date()})")

Train: 3,021 filas (2005-12-29 → 2017-12-28)
Test : 2,009 filas (2018-01-02 → 2025-12-29)


### 5.3 Modelos

- **Regresión cuantílica lineal** (`QuantileRegressor`): la versión lineal, como línea base de ML. La metemos en un `Pipeline` con `StandardScaler`, como en clase.
- **Gradient Boosting cuantílico** (`GradientBoostingRegressor(loss="quantile")`): ensemble de árboles, igual que el Random Forest de clase, pero entrenado **secuencialmente**: cada árbol corrige los errores de los anteriores minimizando la pinball loss. Captura no linealidades como "el cuantil cae mucho más cuando la volatilidad es alta **y** el mercado está en drawdown".

Ajustamos los hiperparámetros con `GridSearchCV` y **`TimeSeriesSplit`**, una validación cruzada que siempre entrena con el pasado y valida con el futuro, usando la pinball loss como métrica.

In [34]:
pinball_scorer = make_scorer(mean_pinball_loss, alpha=ALPHA, greater_is_better=False)

tscv = TimeSeriesSplit(n_splits=5)

# --- Línea base lineal ---
qr_lin = Pipeline([
    ("scaler", StandardScaler()),
    ("qr", QuantileRegressor(quantile=ALPHA, alpha=0.0, solver="highs")),
])
cv_lin = -cross_val_score(qr_lin, X_train, y_train, cv=tscv, scoring=pinball_scorer)
qr_lin.fit(X_train, y_train)

# --- Gradient Boosting cuantílico con búsqueda de hiperparámetros ---
param_grid = {
    "n_estimators": [150, 300],
    "max_depth": [2, 3],
    "min_samples_leaf": [40, 100],
    "learning_rate": [0.05],
}
gs = GridSearchCV(
    GradientBoostingRegressor(loss="quantile", alpha=ALPHA, random_state=SEED),
    param_grid, cv=tscv, scoring=pinball_scorer, n_jobs=-1,
)
gs.fit(X_train, y_train)
gbm = gs.best_estimator_

print(f"Pinball loss CV · QR lineal : {1e4 * cv_lin.mean():.3f} ×1e-4")
print(f"Pinball loss CV · GBM (best): {-1e4 * gs.best_score_:.3f} ×1e-4")
print("Mejores hiperparámetros:", gs.best_params_)

Pinball loss CV · QR lineal : 32.041 ×1e-4
Pinball loss CV · GBM (best): 14.830 ×1e-4
Mejores hiperparámetros: {'learning_rate': 0.05, 'max_depth': 2, 'min_samples_leaf': 100, 'n_estimators': 150}


In [35]:
cv_res = (pd.DataFrame(gs.cv_results_)[["params", "mean_test_score", "std_test_score", "rank_test_score"]]
          .assign(mean_test_score=lambda d: -1e4 * d["mean_test_score"],
                  std_test_score=lambda d: 1e4 * d["std_test_score"])
          .rename(columns={"mean_test_score": "pinball CV (×1e-4)", "std_test_score": "std (×1e-4)"})
          .sort_values("rank_test_score"))
cv_res

,params,pinball CV (×1e-4),std (×1e-4),rank_test_score
2,"{'learning_rate': 0.05, 'max_depth': 2, 'min_s...",14.830203,8.488785,1
3,"{'learning_rate': 0.05, 'max_depth': 2, 'min_s...",14.909048,8.520863,2
7,"{'learning_rate': 0.05, 'max_depth': 3, 'min_s...",14.995940,8.655321,3
6,"{'learning_rate': 0.05, 'max_depth': 3, 'min_s...",15.014658,8.820949,4
0,"{'learning_rate': 0.05, 'max_depth': 2, 'min_s...",15.452728,9.698179,5
4,"{'learning_rate': 0.05, 'max_depth': 3, 'min_s...",15.543725,9.638642,6
5,"{'learning_rate': 0.05, 'max_depth': 3, 'min_s...",15.727917,9.767747,7
1,"{'learning_rate': 0.05, 'max_depth': 2, 'min_s...",15.848477,10.239294,8


In [36]:
test["q_qr"] = qr_lin.predict(X_test)
test["q_gbm"] = gbm.predict(X_test)

### 5.4 Backtesting del VaR

Un VaR del 95 % se valida comprobando sus **violaciones**, los días en que $r_{t+1} < \hat q_{\alpha,t}$:

1. **Cobertura (Kupiec, 1995):** la proporción de violaciones debe ser ≈ 5 %. Bajo $H_0$,
   $LR_{POF} = -2\ln\frac{(1-\alpha)^{n-x}\alpha^x}{(1-\hat p)^{n-x}\hat p^{\,x}} \sim \chi^2_1$, con $\hat p = x/n$.
2. **Independencia (Christoffersen, 1998):** las violaciones no deben agruparse. Si un modelo falla varios días seguidos, está reaccionando tarde a la crisis.
3. **Pinball loss:** la métrica que se optimiza; cuanto menor, mejor.
4. **VaR medio:** el capital que habría que reservar. A igual cobertura, cuanto menor, más eficiente.

Un p-valor > 0.05 significa que **no se rechaza** que el modelo esté bien calibrado.

In [37]:
def kupiec_test(violations, alpha=ALPHA):
    # H0: proporción de violaciones = alpha.  LR ~ chi2(1)
    v = np.asarray(violations, dtype=bool)
    n, x = v.size, v.sum()
    p = np.clip(x / n, 1e-12, 1 - 1e-12)
    ll_h0 = (n - x) * np.log(1 - alpha) + x * np.log(alpha)
    ll_h1 = (n - x) * np.log(1 - p) + x * np.log(p)
    lr = -2 * (ll_h0 - ll_h1)
    return 1 - stats.chi2.cdf(lr, df=1)

def christoffersen_test(violations):
    # H0: una violación hoy no cambia la probabilidad de violación mañana.  LR ~ chi2(1)
    v = np.asarray(violations, dtype=int)
    prev, curr = v[:-1], v[1:]
    n00 = np.sum((prev == 0) & (curr == 0)); n01 = np.sum((prev == 0) & (curr == 1))
    n10 = np.sum((prev == 1) & (curr == 0)); n11 = np.sum((prev == 1) & (curr == 1))
    clip = lambda p: np.clip(p, 1e-12, 1 - 1e-12)
    pi0 = clip(n01 / max(n00 + n01, 1))
    pi1 = clip(n11 / max(n10 + n11, 1))
    pi = clip((n01 + n11) / max(n00 + n01 + n10 + n11, 1))
    ll_h0 = (n00 + n10) * np.log(1 - pi) + (n01 + n11) * np.log(pi)
    ll_h1 = n00 * np.log(1 - pi0) + n01 * np.log(pi0) + n10 * np.log(1 - pi1) + n11 * np.log(pi1)
    lr = -2 * (ll_h0 - ll_h1)
    return 1 - stats.chi2.cdf(lr, df=1)

METHODS = {
    "Histórico (250d)": "q_hist",
    "Normal (250d)": "q_normal",
    "EWMA RiskMetrics": "q_ewma",
    "Quantile Reg. lineal": "q_qr",
    "Quantile GBM": "q_gbm",
}

def backtest(df, methods=METHODS, alpha=ALPHA):
    rows = []
    for name, col in methods.items():
        v = (df["target"] < df[col]).values
        rows.append({
            "Modelo": name,
            "Violaciones": int(v.sum()),
            "Esperadas": round(alpha * len(v), 1),
            "Tasa (%)": 100 * v.mean(),
            "Kupiec p": kupiec_test(v, alpha),
            "Christoffersen p": christoffersen_test(v),
            "Pinball (×1e-4)": 1e4 * mean_pinball_loss(df["target"], df[col], alpha=alpha),
            "VaR medio (%)": -100 * df[col].mean(),
        })
    return pd.DataFrame(rows).set_index("Modelo").round(4)

bt = backtest(test)
bt

,Violaciones,Esperadas,Tasa (%),Kupiec p,Christoffersen p,Pinball (×1e-4),VaR medio (%)
Modelo,,,,,,,
Histórico (250d),114,100.5,5.6745,0.1741,0.0000,15.9781,1.8383
Normal (250d),120,100.5,5.9731,0.0519,0.0000,16.1257,1.8116
EWMA RiskMetrics,115,100.5,5.7242,0.1450,0.1861,13.8898,1.7426
Quantile Reg. lineal,96,100.5,4.7785,0.6464,0.5073,12.9224,1.7508
Quantile GBM,106,100.5,5.2763,0.5732,0.5448,13.0484,1.7008


In [38]:
fig = px.bar(bt.reset_index(), x="Modelo", y="Tasa (%)", color="Modelo", text_auto=".2f",
             title="Tasa de violaciones en test (objetivo: 5 %)")
fig.add_hline(y=100 * ALPHA, line_dash="dash", line_color="black", annotation_text="5 % objetivo")
fig.update_layout(showlegend=False)
fig.show()

In [39]:
# Backtest dentro de las crisis del periodo de test
crisis_bt = pd.concat({
    "COVID (feb–jun 2020)": backtest(test.loc["2020-02-19":"2020-06-30"])[["Violaciones", "Tasa (%)"]],
    "Bear market 2022": backtest(test.loc["2022-01-03":"2022-10-12"])[["Violaciones", "Tasa (%)"]],
}, axis=1)
crisis_bt

COVID (feb–jun 2020)          Bear market 2022         
                              Violaciones Tasa (%)      Violaciones Tasa (%)
Modelo                                                                      
Histórico (250d)                       16  17.2043               23  11.7347
Normal (250d)                          14  15.0538               26  13.2653
EWMA RiskMetrics                        7   7.5269               19   9.6939
Quantile Reg. lineal                   10  10.7527               14   7.1429
Quantile GBM                            9   9.6774               21  10.7143

El backtest por periodos es **donde se ve si un VaR sirve o no**: un modelo puede tener un 5 % de violaciones en el conjunto del test y concentrarlas todas en las semanas de crisis. Eso es exactamente lo que detecta el test de Christoffersen.

In [40]:
line_styles = {"q_normal": ("Normal (250d)", "crimson"), "q_hist": ("Histórico (250d)", "orange"),
               "q_ewma": ("EWMA", "seagreen"), "q_gbm": ("Quantile GBM", "royalblue")}

def plot_var(df, title):
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=df.index, y=df["target"], mode="markers", name="r_{t+1} real",
                             marker=dict(size=3, color="gray")))
    for col, (name, color) in line_styles.items():
        fig.add_trace(go.Scatter(x=df.index, y=df[col], mode="lines", name=name,
                                 line=dict(color=color, width=1.5)))
    viol = df[df["target"] < df["q_gbm"]]
    fig.add_trace(go.Scatter(x=viol.index, y=viol["target"], mode="markers", name="Violación GBM",
                             marker=dict(symbol="x", size=7, color="royalblue")))
    fig.update_layout(title=title, yaxis_tickformat=".0%", yaxis_title="Retorno / cuantil 5 %")
    return fig

plot_var(test, "Backtest del VaR 95 % en test (2018–2025)").update_layout(
    xaxis=dict(rangeslider=dict(visible=True))).show()
plot_var(test.loc["2020-01-15":"2020-07-31"], "Zoom: crash del COVID-19").show()

In [41]:
# Tasa de violaciones móvil (250 sesiones): un buen modelo se mantiene cerca del 5 % en todo momento
roll = pd.DataFrame({name: (test["target"] < test[col]).rolling(250).mean() * 100
                     for name, col in METHODS.items()}).dropna()
fig = px.line(roll, title="Tasa de violaciones móvil (ventana 250 sesiones)")
fig.add_hline(y=100 * ALPHA, line_dash="dash", line_color="black")
fig.update_layout(yaxis_title="% violaciones", xaxis_title=None, legend_title_text="Modelo")
fig.show()

### 5.5 ¿Qué ha aprendido el modelo?

Como en los modelos de árboles vistos en clase, usamos `feature_importances_`: cuánto ha contribuido cada variable a reducir la pérdida en las divisiones de los árboles.

In [42]:
imp = (pd.DataFrame({"Feature": FEATURES, "Importancia": gbm.feature_importances_})
       .sort_values("Importancia"))
fig = px.bar(imp, x="Importancia", y="Feature", orientation="h", text_auto=".3f",
             title="Importancia de variables · Quantile GBM")
fig.show()

In [43]:
# Respuesta del VaR a la volatilidad: el GBM aprende una relación no lineal
test["VaR_gbm"] = -test["q_gbm"]
fig = px.scatter(test, x="vol_20d", y="VaR_gbm", color="drawdown", opacity=0.6,
                 color_continuous_scale="RdBu", labels={"VaR_gbm": "VaR 95 % GBM", "vol_20d": "Vol 20d"},
                 title="VaR predicho vs volatilidad reciente (color = drawdown)")
fig.update_layout(xaxis_tickformat=".1%", yaxis_tickformat=".1%")
fig.show()

## 6. Exportar el modelo

Guardamos con `joblib`, como en clase, el modelo junto con la lista de features y los metadatos que `var_predict.py` necesita.

In [44]:
os.makedirs("models", exist_ok=True)
bundle = {
    "model": gbm,
    "features": FEATURES,
    "alpha": ALPHA,
    "ticker": TICKER,
    "train_end": TRAIN_END,
}
joblib.dump(bundle, os.path.join("models", "var_gbm.pkl"))
print("Modelo guardado en models/var_gbm.pkl")

Modelo guardado en models/var_gbm.pkl


### Uso desde línea de comandos

```bash
python var_predict.py 2020-03-16            # VaR al cierre de esa fecha (cartera de 100.000 €)
python var_predict.py 2020-03-16 250000     # con otro tamaño de cartera
python var_predict.py                       # sin fecha → última sesión disponible
```

In [45]:
!python var_predict.py 2020-03-16 100000

zsh:1: command not found: python


## 7. Conclusiones

1. **Los retornos del S&P 500 no son normales.** Tienen curtosis muy alta, asimetría negativa y muchos más eventos de más de 4σ de los que admite una normal. Cualquier VaR gaussiano con parámetros fijos subestima el riesgo de cola.
2. **El riesgo es predecible aunque la dirección no lo sea.** La ACF de $|r_t|$ muestra memoria larga de la volatilidad. Ahí está la información que puede aprovechar un modelo de ML.
3. **Monte Carlo:** suponer días independientes, aunque tengan colas gruesas, no basta para reproducir los crashes mensuales. El *stress test* muestra que la probabilidad de un crash depende del régimen de mercado.
4. **Backtest:** comparando las tablas de la sección 5.4, el VaR histórico y el normal con ventana de 250 días son los que más concentran violaciones en las crisis, porque reaccionan tarde. Los modelos condicionales (EWMA y quantile GBM) se adaptan al régimen. El quantile GBM lo hace **sin suponer ninguna distribución**, aprendiendo el cuantil directamente con la pinball loss.

**Limitaciones y extensiones:**
- Un solo activo; el siguiente paso sería una cartera multiactivo.
- Solo VaR 95 %; para requisitos regulatorios (Basilea) se usaría VaR 99 % y *Expected Shortfall* 97.5 %.
- El modelo no se reentrena durante el test. En producción se haría un reentrenamiento periódico (*walk-forward*).